# FinBERT Fine-Tuning on MD&A Sentiment (MPS)

**Pipeline**

1. Load `mda_shared_preprocessed.csv` from `sentiment_analysis_data/`
2. Time-based stratified split: filings <= 2022 -> train, 2023+ -> test
3. Fine-tune `ProsusAI/finbert` on Apple Silicon MPS
4. Evaluate on held-out test set (macro F1 + classification report)


## 1. Imports & Device Setup


In [1]:
import warnings
import os

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from datasets import Dataset as HFDataset
from sklearn.metrics import classification_report, f1_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# -- Device (force Apple Silicon MPS) -----------------------------------------
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is not available. Run this notebook on an Apple Silicon Mac with MPS enabled.")
DEVICE = torch.device("mps")
USE_MPS_MIXED_PRECISION = True

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
print(f"MPS mixed precision: {USE_MPS_MIXED_PRECISION}")


PyTorch : 2.8.0
Device  : mps
MPS mixed precision: True


## 2. Load & Prepare Dataset


In [2]:
# Local project root setup (single path)
from pathlib import Path
import os

PROJECT_ROOT = Path('/Users/lunlun/Downloads/Github/textmining_grp6')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project folder not found at {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Current working directory: {Path.cwd()}")

Project root: /Users/lunlun/Downloads/Github/textmining_grp6
Current working directory: /Users/lunlun/Downloads/Github/textmining_grp6


In [3]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

CSV_PATH = Path('/Users/lunlun/Downloads/Github/textmining_grp6/datasets/final/mda_shared_preprocessed.csv')
docs = pd.read_csv(CSV_PATH)
print(f"Loaded: {docs.shape[0]:,} rows from {CSV_PATH}")
docs.head()

Loaded: 452,390 rows from /Users/lunlun/Downloads/Github/textmining_grp6/datasets/final/mda_shared_preprocessed.csv


,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral


In [4]:
print(f"Loaded : {docs.shape[0]:,} documents, {docs.shape[1]} columns")
print(f"Columns: {docs.columns.tolist()}")
print(f"Companies : {docs['company_name'].nunique():,}")
print(f"Year range: {docs['year'].min()} – {docs['year'].max()}")
print()
print("Filing type counts:")
print(docs["filing_type"].value_counts().to_string())


Loaded : 452,390 documents, 8 columns
Columns: ['doc_id', 'company_name', 'filing_type', 'filing_date', 'year', 'quarter', 'sentence', 'sentiment']
Companies : 473
Year range: 2010 – 2025

Filing type counts:
filing_type
10-K    237517
10-Q    214873


In [5]:
docs.head()

,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral


In [6]:
long_df = docs.rename(columns={"sentence": "text"}).copy()
long_df["text"] = long_df["text"].astype(str).str.strip()
long_df = long_df[long_df["text"].str.len() > 0].reset_index(drop=True)

MAX_SENTENCES = 500_000
if MAX_SENTENCES and len(long_df) > MAX_SENTENCES:
    long_df = (
        long_df.groupby("year", group_keys=False)
        .apply(lambda g: g.sample(frac=MAX_SENTENCES / len(long_df), random_state=SEED))
        .reset_index(drop=True)
    )
    print(f"Sampled down to {len(long_df):,} sentences (stratified by year)")

print(f"Documents : {docs.shape[0]:,}")
print(f"Sentences : {len(long_df):,}  (one row each)")
long_df[["doc_id", "company_name", "filing_type", "year", "quarter", "text"]].head(5)

Documents : 452,390
Sentences : 452,390  (one row each)


,doc_id,company_name,filing_type,year,quarter,text
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,presently the companys product line includes a...
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,approximately NUM of all sales were generated ...
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,on an ongoing basis we re-evaluate our judgmen...
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,we base our estimates and judgments on our his...
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,actual results may differ from these estimates...


## 3. FinBERT Architecture Configuration

FinBERT is built on **BERT-base** (`BertConfig`). Below are the four structural hyperparameters you can define — everything else is fixed by the pre-trained checkpoint.

| Parameter             | Default (FinBERT) | What it controls                                                                 |
| --------------------- | ----------------- | -------------------------------------------------------------------------------- |
| `num_hidden_layers`   | 12                | Number of Transformer encoder blocks stacked on top of each other                |
| `num_attention_heads` | 12                | Parallel attention "heads" per layer — each learns different token relationships |
| `hidden_size`         | 768               | Width of every hidden vector — **must be divisible by `num_attention_heads`**    |
| `intermediate_size`   | 3072              | Inner width of the feed-forward sub-layer (conventionally 4 × `hidden_size`)     |

**Total parameter count** scales roughly as:
`params ≈ num_layers × (4 × hidden² + 2 × hidden × intermediate)`

> **Important**: If you change `num_hidden_layers` / `hidden_size` / `num_attention_heads` away from the defaults, the pre-trained weights **cannot be loaded** for those layers — training starts from random. The values below match the original FinBERT so pre-trained weights are fully reused.


In [7]:
MODEL_NAME = "ProsusAI/finbert"

NUM_HIDDEN_LAYERS = 12
NUM_ATTENTION_HEADS = 12
HIDDEN_SIZE = 768
INTERMEDIATE_SIZE = 3072

LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
MAX_LEN = 96  # shorter sequences speed up both training and inference

# Freeze lower encoder blocks to train much faster on MPS.
FREEZE_LOWER_LAYERS = True
NUM_FROZEN_LAYERS = 8

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    ignore_mismatched_sizes=True,
).to(DEVICE)

if FREEZE_LOWER_LAYERS:
    model.bert.embeddings.requires_grad_(False)
    for layer in model.bert.encoder.layer[:NUM_FROZEN_LAYERS]:
        layer.requires_grad_(False)

if DEVICE.type == "mps" and USE_MPS_MIXED_PRECISION:
    model = model.to(torch.float16)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"Architecture : {NUM_HIDDEN_LAYERS} layers x {NUM_ATTENTION_HEADS} heads x {HIDDEN_SIZE}d hidden"
)
print(f"Total params : {total_params:,}")
print(f"Trainable    : {trainable:,}")
print(f"Device       : {next(model.parameters()).device}")
print(f"Model dtype  : {next(model.parameters()).dtype}")
print(f"Frozen lower layers: {FREEZE_LOWER_LAYERS} (N={NUM_FROZEN_LAYERS})")


Architecture : 12 layers x 12 heads x 768d hidden
Total params : 109,484,547
Trainable    : 28,944,387
Device       : mps:0
Model dtype  : torch.float16
Frozen lower layers: True (N=8)


## 5. Time-Stratified Train / Test Split

Filings **≤ 2022** → train, **2023+** → test.  
This prevents look-ahead bias — the model is evaluated only on filings filed after the training period.


In [ ]:
import os
from pathlib import Path

TRAIN_CUTOFF = 2022  # filings up to and including 2022 -> train; 2023+ -> test

# -- Label encoding (LABEL2ID defined in section 3 architecture cell) ----------
long_df["label"] = long_df["sentiment"].str.strip().str.lower()
long_df = long_df[long_df["label"].isin(LABEL2ID)].reset_index(drop=True)
long_df["label_id"] = long_df["label"].map(LABEL2ID)

train_df = long_df[long_df["year"] <= TRAIN_CUTOFF].copy().reset_index(drop=True)
test_df = long_df[long_df["year"] > TRAIN_CUTOFF].copy().reset_index(drop=True)

print(f"Train (<= {TRAIN_CUTOFF}) : {len(train_df):>8,} sentences")
print(f"Test  ({TRAIN_CUTOFF + 1}+)  : {len(test_df):>8,} sentences")
print()
print("Train label distribution:")
print(train_df["label"].value_counts().to_string())
print()
print("Test label distribution:")
print(test_df["label"].value_counts().to_string())

# -- Tokenize ------------------------------------------------------------------
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

# Use all available CPU workers (minus one for system responsiveness).
N_PROC = max(1, (os.cpu_count() or 1) - 1)

CACHE_DIR = Path("./hf_cache")
TRAIN_CACHE = CACHE_DIR / "train"
TEST_CACHE = CACHE_DIR / "test"

if TRAIN_CACHE.exists() and TEST_CACHE.exists():
    from datasets import load_from_disk

    hf_train = load_from_disk(str(TRAIN_CACHE))
    hf_test = load_from_disk(str(TEST_CACHE))
    print("Loaded tokenized datasets from cache.")
else:
    hf_train = HFDataset.from_dict(
        {"text": train_df["text"].tolist(), "labels": train_df["label_id"].tolist()}
    )
    hf_test = HFDataset.from_dict(
        {"text": test_df["text"].tolist(), "labels": test_df["label_id"].tolist()}
    )
    hf_train = hf_train.map(
        tokenize,
        batched=True,
        batch_size=10_000,
        num_proc=N_PROC,
        remove_columns=["text"],
    )
    hf_test = hf_test.map(
        tokenize,
        batched=True,
        batch_size=10_000,
        num_proc=N_PROC,
        remove_columns=["text"],
    )
    hf_train.save_to_disk(str(TRAIN_CACHE))
    hf_test.save_to_disk(str(TEST_CACHE))
    print("Tokenized and cached to ./hf_cache/")

print(f"\nTokenization workers: {N_PROC}")
print(f"HF train size : {len(hf_train):,}")
print(f"HF test  size : {len(hf_test):,}")


Train (≤ 2022) :  341,359 sentences
Test  (2023+)  :  111,031 sentences

Train label distribution:
label
neutral     284435
positive     29097
negative     27827

Test label distribution:
label
neutral     93951
negative     8613
positive     8467


Saving the dataset (1/1 shards): 100%|██████████| 111031/111031 [00:00<00:00, 3073446.06 examples/s]

Tokenised and cached to ./hf_cache/

HF train size : 341,359
HF test  size : 111,031


## 6. Fine-Tune FinBERT


In [ ]:
import os
import numpy as np
from sklearn.metrics import f1_score
from pathlib import Path
import torch

CHECKPOINT_DIR = Path("./finbert_model")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


def _is_oom_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return "out of memory" in msg or "mps backend out of memory" in msg


def _clear_device_cache():
    if DEVICE.type == "mps" and hasattr(torch.mps, "empty_cache"):
        torch.mps.empty_cache()


def _probe_max_batch_size(
    dataset,
    collator,
    candidates,
    train_mode=True,
):
    best = None
    for bs in candidates:
        if len(dataset) < bs:
            continue
        try:
            batch = collator([dataset[i] for i in range(bs)])
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            model.zero_grad(set_to_none=True)
            if train_mode:
                model.train()
                out = model(**batch)
                out.loss.backward()
                model.zero_grad(set_to_none=True)
            else:
                model.eval()
                with torch.no_grad():
                    _ = model(**batch)
            best = bs
            _clear_device_cache()
        except RuntimeError as exc:
            _clear_device_cache()
            if _is_oom_error(exc):
                break
            raise
    return best


if CHECKPOINT_DIR.exists() and (CHECKPOINT_DIR / "config.json").exists():
    print(f"Checkpoint found at {CHECKPOINT_DIR.resolve()}")
    print("Skipping training - run Section 10 (Load Checkpoint) to restore model.")
else:
    collator = DataCollatorWithPadding(tokenizer)

    # Ascertain safe batch sizes on current device before training.
    train_bs_candidates = [32, 48, 64, 80, 96, 112, 128]
    eval_bs_candidates = [128, 192, 256, 320, 384, 512]

    tuned_train_bs = _probe_max_batch_size(
        hf_train, collator, train_bs_candidates, train_mode=True
    ) or 32
    tuned_eval_bs = _probe_max_batch_size(
        hf_test, collator, eval_bs_candidates, train_mode=False
    ) or max(128, tuned_train_bs * 2)

    # MPS multiprocessing DataLoader has a known PyTorch bug where worker
    # cleanup raises AttributeError: '_workers_status'. Use 0 workers on MPS
    # — data is pre-tokenized Arrow files so there is no IO wait anyway.
    dataloader_workers = 0 if DEVICE.type == "mps" else max(1, (os.cpu_count() or 1) - 1)
    persistent_workers = dataloader_workers > 0

    # Gradient accumulation: simulate 4x effective batch size without extra memory.
    GRAD_ACCUM = 4

    print(f"Auto-tuned train batch size : {tuned_train_bs} (effective {tuned_train_bs * GRAD_ACCUM})")
    print(f"Auto-tuned eval batch size  : {tuned_eval_bs}")
    print(f"Dataloader workers          : {dataloader_workers}")
    print(f"Gradient accumulation steps : {GRAD_ACCUM}")

    training_args = TrainingArguments(
        output_dir="./finbert_mda_checkpoints",
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_ratio=0.1,
        per_device_train_batch_size=tuned_train_bs,
        per_device_eval_batch_size=tuned_eval_bs,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=6,
        fp16=False,
        bf16=False,
        group_by_length=True,
        optim="adamw_torch",
        dataloader_num_workers=dataloader_workers,
        dataloader_pin_memory=False,
        dataloader_persistent_workers=persistent_workers,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=200,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=hf_train,
        eval_dataset=hf_test,
        tokenizer=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=1,
                early_stopping_threshold=5e-4,
            )
        ],
    )

    print("Starting fine-tuning with early stopping...")
    trainer.train()


## 8. Evaluation on Test Set


In [ ]:
model.eval()
preds_out = trainer.predict(hf_test)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids

pred_labels = [ID2LABEL[p] for p in y_pred]
true_labels = [ID2LABEL[t] for t in y_true]

print("=== Fine-tuned FinBERT on MD&A - Test Set (2023+) ===")
print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=["positive", "neutral", "negative"],
        zero_division=0,
    )
)

macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Macro F1: {macro_f1:.4f}")


=== Fine-tuned FinBERT on MD&A — Test Set (2023+) ===
              precision    recall  f1-score   support

    positive       0.88      0.88      0.88      4575
     neutral       0.98      0.98      0.98     49743
    negative       0.87      0.87      0.87      4661

    accuracy                           0.96     58979
   macro avg       0.91      0.91      0.91     58979
weighted avg       0.96      0.96      0.96     58979

Macro F1: 0.9091


## 9. Save Trained Checkpoint

Saves model weights, tokenizer, and label mappings to `./finbert_model/`.  
On the **next run**, the training cell auto-detects this checkpoint and skips training.

In [ ]:
from pathlib import Path
import json

CHECKPOINT_DIR = Path("./finbert_model")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(CHECKPOINT_DIR))
tokenizer.save_pretrained(str(CHECKPOINT_DIR))

# Save metadata alongside weights so the load cell is self-contained
with open(CHECKPOINT_DIR / "label_mappings.json", "w") as f:
    json.dump(
        {
            "label2id": LABEL2ID,
            "id2label": {str(k): v for k, v in ID2LABEL.items()},
            "model_name": MODEL_NAME,
            "max_len": MAX_LEN,
        },
        f,
        indent=2,
    )

print(f"\u2705 Model + tokenizer + metadata saved \u2192 {CHECKPOINT_DIR.resolve()}")
print("   Contents:", [p.name for p in sorted(CHECKPOINT_DIR.iterdir())])


NameError: name 'trainer' is not defined

## 10. Load Trained Checkpoint

**Re-run entry point** — if `./finbert_model/` already exists, run **Sections 1\u20134** (imports, device, data load), then jump straight here.  Skip Sections 5\u20139 entirely.


In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CHECKPOINT_DIR = Path("./finbert_model")

if not (CHECKPOINT_DIR / "config.json").exists():
    raise FileNotFoundError(
        f"No checkpoint at {CHECKPOINT_DIR.resolve()}\n"
        "Run Sections 6\u20139 to train and save first."
    )

# Restore label mappings and hyperparams saved alongside the weights
with open(CHECKPOINT_DIR / "label_mappings.json") as f:
    _meta = json.load(f)

LABEL2ID = _meta["label2id"]
ID2LABEL = {int(k): v for k, v in _meta["id2label"].items()}
MAX_LEN  = _meta["max_len"]

tokenizer = AutoTokenizer.from_pretrained(str(CHECKPOINT_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(CHECKPOINT_DIR)).to(DEVICE)
model.eval()

print(f"\u2705 Checkpoint loaded from {CHECKPOINT_DIR.resolve()}")
print(f"   Labels  : {ID2LABEL}")
print(f"   Max len : {MAX_LEN}")


## 11. Run Inference on Full Dataset

In [ ]:
import torch
import numpy as np
from datasets import Dataset as HFDataset
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

# ── Build inference input from original CSV ────────────────────────────────
infer_df = (
    docs[["doc_id", "company_name", "quarter", "sentence"]]
    .copy()
    .pipe(lambda d: d[d["sentence"].astype(str).str.strip().str.len() > 0])
    .reset_index(drop=True)
)
print(f"Running inference on {len(infer_df):,} sentences...")

# Tokenise
def _tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

infer_hf = HFDataset.from_dict({"text": infer_df["sentence"].tolist()})
infer_hf = infer_hf.map(_tok, batched=True, batch_size=10_000, remove_columns=["text"])

# MPS does not support pin_memory or multiprocessing DataLoader workers.
_infer_args = TrainingArguments(
    output_dir="./finbert_inference_tmp",
    per_device_eval_batch_size=256,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    fp16=False,
    report_to="none",
)
logits = Trainer(
    model=model,
    args=_infer_args,
    data_collator=DataCollatorWithPadding(tokenizer),
).predict(infer_hf).predictions

# Probabilities + labels
probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
infer_df["pos"]             = probs[:, LABEL2ID["positive"]]
infer_df["neg"]             = probs[:, LABEL2ID["negative"]]
infer_df["sentiment_score"] = infer_df["pos"] - infer_df["neg"]
infer_df["label"]           = [ID2LABEL[i] for i in probs.argmax(axis=1)]

display(
    infer_df[["doc_id", "company_name", "quarter", "sentence",
              "label", "pos", "neg", "sentiment_score"]].head()
)


In [ ]:
from pathlib import Path

# ── Final schema ──────────────────────────────────────────────────────────────
# doc_id | company | quarter | sentence | label | pos | neg | sentiment_score
# sentence_id = original CSV row index, shared with BERTopic sent_chunk_map

labeled = (
    infer_df
    .rename(columns={"company_name": "company"})
    [["doc_id", "company", "quarter", "sentence",
      "label", "pos", "neg", "sentiment_score"]]
    .copy()
)
labeled["sentence_id"] = labeled.index  # bridge to sent_chunk_map

OUT_PATH = Path("../webapp/labeled_sentiment.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
labeled.to_parquet(OUT_PATH, index=False)

print(f"\u2705 {len(labeled):,} rows \u2192 {OUT_PATH.resolve()}")
print(f"   Positive: {(labeled['label']=='positive').sum():,}  "
      f"Neutral: {(labeled['label']=='neutral').sum():,}  "
      f"Negative: {(labeled['label']=='negative').sum():,}")
display(labeled.head())
